In [ ]:
# -*- coding: utf-8 -*-
"""
LoRA Baseline - CIFAR-100

Standard LoRA implementation for comparison with CDWF.

Tests multiple LoRA ranks: 1, 2, 4, 8, 16
"""

import os
import gc
import math
import random
import time
import copy

from tqdm import tqdm
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import OneCycleLR

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


# ================================================================
# GPU POWER MONITORING
# ================================================================
try:
    import pynvml

    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print("GPU power monitoring enabled")
except Exception:
    NVML_AVAILABLE = False
    _nvml_handle = None
    print("GPU power monitoring unavailable")


def read_gpu_power_w():
    if not NVML_AVAILABLE:
        return 0.0
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_nvml_handle) / 1000.0
    except Exception:
        return 0.0


# ================================================================
# CONFIGURATION
# ================================================================
EXPERIMENT_NAME = "cifar100_resnet50_lora_baseline_10ep"
SEED = 42

# Model
MODEL_DEPTH = 50
USE_PRETRAINED = True

# Training
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 10
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0

# LoRA configuration
LORA_RANKS = [1, 2, 4, 8, 16]
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# Data
NUM_CLASSES = 100
DATA_DIR = "./data"
FULLFT_CSV = "cifar100_resnet50_fullft_results.csv"
RESULT_CSV = "cifar100_resnet50_lora_baseline_results.csv"

print(f"\n{'=' * 70}")
print("  LORA BASELINE - CIFAR-100")
print(f"{'=' * 70}")
print(f"Model: ResNet-{MODEL_DEPTH} (ImageNet pretrained)")
print(f"Testing LoRA ranks: {LORA_RANKS}")
print(f"LoRA alpha: {LORA_ALPHA}")
print(f"Training epochs: {EPOCHS}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"Classes: {NUM_CLASSES}")
print(f"{'=' * 70}\n")


# ================================================================
# REPRODUCIBILITY
# ================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")


# ================================================================
# DATA LOADING
# ================================================================
print("Loading CIFAR-100...")

mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset_full = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform_train,
)
test_dataset = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform_test,
)

train_size = int(0.80 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Train: {train_size}, Val: {val_size}, Test: {len(test_dataset)}\n")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ================================================================
# MODEL
# ================================================================
def get_resnet_cifar100(depth=50, pretrained=True):
    model_fns = {
        18: models.resnet18,
        50: models.resnet50,
        101: models.resnet101,
    }
    model = model_fns[depth](pretrained=pretrained)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


# ================================================================
# LORA IMPLEMENTATION
# ================================================================
class LoRALayer(nn.Module):
    """Standard LoRA wrapper for Conv2d."""

    def __init__(self, base_layer, rank=4, alpha=16, dropout=0.0):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha

        for param in self.base_layer.parameters():
            param.requires_grad = False

        if isinstance(base_layer, nn.Conv2d):
            in_channels = base_layer.in_channels
            out_channels = base_layer.out_channels
            kernel_size = base_layer.kernel_size
            stride = base_layer.stride
            padding = base_layer.padding

            self.lora_A = nn.Conv2d(
                in_channels,
                rank,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False,
            )
            self.lora_B = nn.Conv2d(
                rank,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False,
            )

            nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B.weight)

            self.dropout = nn.Dropout(dropout) if dropout > 0 else None
            self.scaling = alpha / rank

    def forward(self, x):
        base_out = self.base_layer(x)

        lora_out = self.lora_A(x)
        if self.dropout is not None:
            lora_out = self.dropout(lora_out)
        lora_out = self.lora_B(lora_out)

        return base_out + self.scaling * lora_out


def add_lora_to_model(model, rank=4, alpha=16, dropout=0.0):
    """Add LoRA to conv2 layers in ResNet blocks."""
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for block in layer:
            if hasattr(block, "conv2"):
                block.conv2 = LoRALayer(block.conv2, rank, alpha, dropout)

    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


def freeze_base_model(model):
    """Freeze all parameters except LoRA layers and classifier."""
    for name, param in model.named_parameters():
        if "lora" not in name and not name.startswith("fc"):
            param.requires_grad = False


# ================================================================
# TRAINING FUNCTIONS
# ================================================================
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler, device, power_log=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Train", leave=False):
        if power_log is not None:
            now = time.time()
            power_log["energy_J"] += read_gpu_power_w() * (now - power_log["last_t"])
            power_log["last_t"] = now

        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        if CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in tqdm(loader, desc="Eval", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


# ================================================================
# LOAD FULL_FT BASELINE FROM CSV
# ================================================================
print("=" * 70)
print("  LOADING CIFAR-100 FULL_FT BASELINE")
print("=" * 70 + "\n")

if os.path.exists(FULLFT_CSV):
    print(f"Loading baseline from: {FULLFT_CSV}")
    fullft_df = pd.read_csv(FULLFT_CSV)
    fullft_test_acc = fullft_df["test_acc"].iloc[0]
    fullft_val_acc = fullft_df["val_acc"].iloc[0]
    fullft_time = fullft_df["training_time_s"].iloc[0]
    fullft_energy = fullft_df["energy_J"].iloc[0]

    print("Loaded CIFAR-100 Full_FT baseline")
    print(f"  Val: {fullft_val_acc:.2f}%")
    print(f"  Test: {fullft_test_acc:.2f}%")
    print(f"  Training Time: {fullft_time:.1f}s")
    print(f"  Energy: {fullft_energy / 1000:.2f}kJ\n")
else:
    print("Error: CIFAR-100 full fine-tuning baseline not found.")
    print(f"Expected file: {FULLFT_CSV}")
    print("Run the CIFAR-100 full fine-tuning script first.\n")
    raise SystemExit(1)


# ================================================================
# TRAIN LORA WITH DIFFERENT RANKS
# ================================================================
all_results = []

for lora_rank in LORA_RANKS:
    print("=" * 70)
    print(f"  LORA RANK {lora_rank}")
    print("=" * 70 + "\n")

    torch.cuda.empty_cache()
    gc.collect()
    set_seed(SEED)

    print(f"Creating ResNet-{MODEL_DEPTH} with LoRA rank-{lora_rank}...")
    model = get_resnet_cifar100(MODEL_DEPTH, USE_PRETRAINED)

    print("Model initialized with ImageNet weights\n")

    model = add_lora_to_model(
        model,
        rank=lora_rank,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT,
    )

    freeze_base_model(model)
    model = model.to(device)

    total_params, trainable_params, frozen_params = count_params(model)
    classifier_params = sum(p.numel() for p in model.fc.parameters())
    lora_params = trainable_params - classifier_params

    print("[Model Summary]")
    print(f"  Total params: {total_params:,} ({total_params / 1e6:.2f}M)")
    print(f"  Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"  LoRA params: {lora_params:,}")
    print(f"  Reduction: {total_params / trainable_params:.1f}x\n")

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable, lr=MAX_LR, weight_decay=WEIGHT_DECAY)
    scheduler = OneCycleLR(
        optimizer,
        max_lr=MAX_LR,
        steps_per_epoch=len(train_loader),
        epochs=EPOCHS,
        pct_start=0.15,
        div_factor=10,
        final_div_factor=100,
    )

    print(f"Training for {EPOCHS} epochs...\n")

    start_time = time.time()
    power_log = {"energy_J": 0.0, "last_t": start_time}

    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            power_log=power_log,
        )
        val_loss, val_acc = evaluate(model, val_loader, device)

        print(f"Epoch {epoch}/{EPOCHS}: train_acc={train_acc:.2f}% val_acc={val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    training_time = time.time() - start_time
    training_energy = power_log["energy_J"]

    if best_state is not None:
        model.load_state_dict(best_state)

    _, test_acc = evaluate(model, test_loader, device)

    retention = test_acc / fullft_test_acc
    param_reduction = total_params / trainable_params
    time_speedup = fullft_time / training_time
    energy_reduction = fullft_energy / training_energy

    print(f"\n{'=' * 70}")
    print(f"  RESULTS: LORA RANK {lora_rank}")
    print(f"{'=' * 70}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Retention: {retention * 100:.1f}%")
    print(f"Trainable Params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"Parameter Reduction: {param_reduction:.1f}x\n")
    print(f"Training Time: {training_time:.1f}s ({training_time / 60:.2f} min)")
    print(f"Time Speedup: {time_speedup:.2f}x")
    print(f"Energy: {training_energy / 1000:.2f} kJ")
    print(f"Energy Reduction: {energy_reduction:.2f}x")
    print(f"{'=' * 70}\n")

    results = {
        "method": "LoRA",
        "dataset": "cifar100",
        "lora_rank": lora_rank,
        "lora_alpha": LORA_ALPHA,
        "fullft_test_acc": fullft_test_acc,
        "fullft_val_acc": fullft_val_acc,
        "test_acc": test_acc,
        "retention": retention,
        "trainable_params": trainable_params,
        "trainable_percent": 100 * trainable_params / total_params,
        "param_reduction": param_reduction,
        "training_time_s": training_time,
        "time_speedup": time_speedup,
        "energy_J": training_energy,
        "energy_kJ": training_energy / 1000,
        "energy_reduction": energy_reduction,
    }

    all_results.append(results)


# ================================================================
# SAVE ALL RESULTS
# ================================================================
print("=" * 70)
print("  SUMMARY: ALL LORA CONFIGURATIONS")
print("=" * 70 + "\n")

df = pd.DataFrame(all_results)
print(df.to_string(index=False))
print()

if os.path.exists(RESULT_CSV):
    existing_df = pd.read_csv(RESULT_CSV)
    df = pd.concat([existing_df, df], ignore_index=True)

df.to_csv(RESULT_CSV, index=False)
print(f"Results saved to {RESULT_CSV}\n")

print("=" * 70)
print("  COMPARISON: CIFAR-100 FULL_FT VS LORA")
print("=" * 70)
print(f"{'Rank':<8} {'Retention':<12} {'Speedup':<10} {'Energy':<10} {'Params':<10}")
print("-" * 70)
print(f"{'Full_FT':<8} {'100.0%':<12} {'1.00x':<10} {'1.00x':<10} {'100.0%':<10}")
for result in all_results:
    print(
        f"r={result['lora_rank']:<5} {result['retention'] * 100:>6.1f}%     "
        f"{result['time_speedup']:>6.2f}x    {result['energy_reduction']:>6.2f}x    "
        f"{result['trainable_percent']:>6.2f}%"
    )
print("=" * 70 + "\n")

print("LoRA Baseline Complete (CIFAR-100)")
print(f"Tested ranks: {LORA_RANKS}")
print(f"Best retention: {max(r['retention'] for r in all_results) * 100:.1f}%")
print(f"Best speedup: {max(r['time_speedup'] for r in all_results):.2f}x")